In [1]:
import numpy as np
from FallbackGen import FallbackGen
from TDECalculator_edited import TDECalculator
import gc

In [2]:
MBH = 1e4
Rp = 3.645
a = -0.2
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [6]:
import numpy as np

rp = 3.645
a = -0.2
E = 1.0
Q = 0.0

# Retrograde Kerr angular momentum branch
sqrt_term = np.sqrt(
    2 * rp**3 * (a**2 + (rp - 2) * rp)
)

Lz = (
    -2 * a * rp - sqrt_term
) / ((rp - 2) * rp)

print("Lz =", Lz)


def R_of_r(r):
    delta = r**2 - 2*r + a**2

    return (
        (E * (r**2 + a**2) - a * Lz)**2
        - delta * (
            r**2 + (Lz - a * E)**2 + Q
        )
    )


for r in [3.645, 3.646, 3.65, 3.7, 4.0, 5.0, 6.0, 10.0]:
    print(f"r = {r:.3f}, R = {R_of_r(r): .8e}")

Lz = -3.7893296275360235
r = 3.645, R =  0.00000000e+00
r = 3.646, R =  8.12988631e-04
r = 3.650, R =  4.21540277e-03
r = 3.700, R =  6.73546282e-02
r = 4.000, R =  1.32199298e+00
r = 5.000, R =  1.98573961e+01
r = 6.000, R =  6.96747612e+01
r = 10.000, R =  8.21763841e+02


In [3]:
mass_r = TDECalculator('CO_WD_3Msun_z0p0142', orbit, MBH, Rp, a, N=N)

In [4]:
sample_r = mass_r.rel_whole_star_sample()

In [5]:
radii_r = sample_r['rr']

rtde = mass_r.R_TDE
Lz = mass_r.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_r <= 0.5,
    rtde - radii_r * mass_r.Rstar,
    rtde + radii_r * mass_r.Rstar
)

deltaE = mass_r.Rstar / mass_r.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_r['dEnergy_random'] * deltaE 
dLz = mass_r.dLz_random
mass_ratio = mass_r.mass_ratio

In [6]:
dT_r = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, dE, dLz, N)
np.save('dT_r.npy', dT_r.dTs)
gc.collect()

Computing radial periods for 113,670,000 particles ...
  E  range: [0.980099, 1.019900]
  Q  range: [0.000e+00, 0.000e+00]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 11399/11399 (100%)
  Bound: 56,993,561 / 113,670,000
  Valid roots: 17,101,517
  Valid Lambda_r: 17,101,517
  Quadrature: 343 chunks ...
    chunk 343/343  (100%)
  Successful T_r: 17,101,517 / 113,670,000


0

In [7]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample["dEnergy_random"]
    dT_rand = dT / delta
    dMass = whole_star_sample["dMass"]

    bins_E = np.linspace(-2.0, 2.0, 1000)
    bins_T = np.logspace(0.0, 6.0, 1000)

    total_mass = np.sum(dMass)
    if not np.isfinite(total_mass) or total_mass <= 0.0:
        raise ValueError("Sample has non-finite or non-positive total mass")

    valid_E = (
        np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dE_rand >= bins_E[0])
        & (dE_rand <= bins_E[-1])
    )
    mass_E, edges_E = np.histogram(
        dE_rand[valid_E],
        bins=bins_E,
        weights=dMass[valid_E] / total_mass,
        density=False,
    )
    hist_E = mass_E / np.diff(edges_E)

    valid_T = (
        np.isfinite(dT_rand)
        & np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dT_rand >= bins_T[0])
        & (dT_rand <= bins_T[-1])
    )
    mass_T, edges_T = np.histogram(
        dT_rand[valid_T],
        bins=bins_T,
        weights=dMass[valid_T] / total_mass,
        density=False,
    )
    hist_T = mass_T / np.diff(edges_T)

    energy_fraction = np.sum(dMass[valid_E]) / total_mass
    returning_fraction = np.sum(dMass[valid_T]) / total_mass
    assert np.isclose(
        np.sum(hist_E * np.diff(edges_E)), energy_fraction, rtol=1e-12
    )
    assert np.isclose(
        np.sum(hist_T * np.diff(edges_T)), returning_fraction, rtol=1e-12
    )

    print(
        "mass fractions: "
        f"energy range={energy_fraction:.6f}, "
        f"returning/time range={returning_fraction:.6f}"
    )

    energy_plot = {
        "x": 0.5 * (edges_E[:-1] + edges_E[1:]),
        "y": hist_E,
    }
    fallback_plot = {
        "x": 0.5 * (edges_T[:-1] + edges_T[1:]),
        "y": hist_T,
    }
    return energy_plot, fallback_plot

In [8]:
DeltaE = mass_r.Rstar / mass_r.Rp**2
DeltaT = 1 / DeltaE**1.5

In [9]:
rel_r_E, rel_r_T = make_plot_dicts(sample_r, dT_r.dTs, DeltaT)

mass fractions: energy range=1.000000, returning/time range=0.125518


In [10]:
import json

adden = "WD3SM-m1_rp3p645_a0p2"

with open(f"Fallback_Data/rel_r_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_E.items()}, f)
with open(f"Fallback_Data/rel_r_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_T.items()}, f)